In [ ]:

import sys
from pathlib import Path
BASE_PATH = Path.cwd().absolute()

sys.path.append(str(BASE_PATH))
sys.path.append(str(BASE_PATH / "Studies"))
sys.path.append(str(BASE_PATH / "Functions"))


import Gandalf as Gan_model
import MLP as MLP_model
import Confinv
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from pytorch_tabnet.tab_model import TabNetClassifier
import xgboost as xgb


import pandas as pd
import numpy as np
import torch.nn as nn
import torch
import sqlalchemy as sa
import Results as res
import Get_Data as gd
import time
from sklearn.model_selection import train_test_split

import torch

from torch.utils.data import DataLoader
import torch.utils.data as data_utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cpu=torch.device("cpu")

/home/jrech/predictive-goods-receipt-scheduling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda
gpu: NVIDIA GeForce RTX 4060 Laptop GPU
Using device: cuda
device: cuda
gpu: NVIDIA GeForce RTX 4060 Laptop GPU
device: cuda
gpu: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
path="Database/DB_params.db"
engine = sa.create_engine("sqlite:///" + path)
engine.connect()


path_train="Training_Test_Datensatz/Training_Datensatz.xlsx"
path_test="Training_Test_Datensatz/Test_Datensatz.xlsx"


Train= pd.read_excel(path_train)
Test= pd.read_excel(path_test)

X_test_raw = Test.drop(columns=["verzoegerung_bin_5","w_time","Verzoegerungstage"])
y_test_raw = Test["verzoegerung_bin_5"]
w_test_raw = Test["w_time"]

X_train_raw = Train.drop(columns=["verzoegerung_bin_5","w_time","Verzoegerungstage"])
y_train_raw = Train["verzoegerung_bin_5"]
w_train_raw = Train["w_time"]



In [3]:
conf=Confinv.ConfidenceIntervalCalculator()

In [ ]:
Model="Logistic Regression"

best_params=pd.read_sql(Model, engine).iloc[0].to_dict()


if best_params["class_weight"]=="balanced":
    best_params["class_weight"] = "balanced"
else:
    best_params["class_weight"] = None

X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer"))
model = LogisticRegression(
    solver=best_params["solver"],
    C=best_params["C"],
    max_iter=best_params["max_iter"],
    class_weight=best_params["class_weight"],
    random_state=42
)



start=time.time()
model.fit(X_train, y_train, sample_weight=w_train)
end=time.time()


X_test = np.nan_to_num(X_test, nan=0)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)

res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

Results for Logistic Regression saved successfully at 2026-06-03 11:23.


In [ ]:

Model="CatBoost"
best_params=pd.read_sql(Model, engine).iloc[0].to_dict()
X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw, scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer"))
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)
best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

final_model = CatBoostClassifier(**best_params, allow_writing_files=False)

start=time.time()
final_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    sample_weight=w_tr,
    early_stopping_rounds=50,
    verbose=False
)
end=time.time()
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)
traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test

)
res.save_frames(Model, y_pred.flatten(), y_proba ,y_test, w_test, training_time=traing_time)



Results for CatBoost saved successfully at 2026-06-03 11:24.


In [ ]:

Model="Gandalf"

epochs=100
loss_fn = nn.CrossEntropyLoss(reduction="none")
best_params=pd.read_sql(Model, engine).iloc[0].to_dict()
X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw, scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer") , feature_output="MLP")
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)


best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

batch_size=best_params.pop("batch_size")

train_dataset = data_utils.TensorDataset(X_tr, y_tr, w_tr)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataset = data_utils.TensorDataset(X_val, y_val, w_val)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

model = Gan_model.Model(
                    input_dim=X_train.shape[1],
                    gflu_stages=best_params["gflu_stages"],
                    gflu_dropout=best_params["gflu_dropout"],
                    gflu_feature_init_sparsity=best_params["gflu_feature_init_sparsity"],
                    learnable_sparsity=True,
                    )

optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=best_params.pop("weight_decay"))
sheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, min_lr=1e-6)


start=time.time()
accuracay_plot=model.fit(train_dataloader, val_dataloader,epochs, optimizer, sheduler, loss_fn)
end=time.time()

y_pred=model.predict(X_test)
y_proba=model.predict_proba(X_test)


conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)

traing_time=end-start
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

Epochs 1, Loss: 1.4461, Val Loss: 1.3641, Val Acc: 0.4586
Epochs 2, Loss: 1.2996, Val Loss: 1.3050, Val Acc: 0.4726
Epochs 3, Loss: 1.2401, Val Loss: 1.2633, Val Acc: 0.4912
Epochs 4, Loss: 1.1895, Val Loss: 1.2194, Val Acc: 0.5022
Epochs 5, Loss: 1.1459, Val Loss: 1.1889, Val Acc: 0.5142
Epochs 6, Loss: 1.1110, Val Loss: 1.1567, Val Acc: 0.5468
Epochs 7, Loss: 1.0743, Val Loss: 1.1299, Val Acc: 0.5439
Epochs 8, Loss: 1.0422, Val Loss: 1.1057, Val Acc: 0.5463
Epochs 9, Loss: 1.0081, Val Loss: 1.0770, Val Acc: 0.5746
Epochs 10, Loss: 0.9772, Val Loss: 1.0541, Val Acc: 0.5855
Epochs 11, Loss: 0.9477, Val Loss: 1.0483, Val Acc: 0.5734
Epochs 12, Loss: 0.9230, Val Loss: 1.0186, Val Acc: 0.5963
Epochs 13, Loss: 0.9002, Val Loss: 0.9823, Val Acc: 0.6138
Epochs 14, Loss: 0.8755, Val Loss: 0.9839, Val Acc: 0.6232
Epochs 15, Loss: 0.8584, Val Loss: 0.9627, Val Acc: 0.6238
Epochs 16, Loss: 0.8383, Val Loss: 0.9706, Val Acc: 0.6194
Epochs 17, Loss: 0.8177, Val Loss: 0.9378, Val Acc: 0.6289
Epochs

In [ ]:
Model="LightGBM"

best_params=pd.read_sql(Model, engine).iloc[0].to_dict()

X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer"))
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)
best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

model = LGBMClassifier(**best_params)

start=time.time()
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    sample_weight=w_tr,
    eval_sample_weight=[w_val]
)
end=time.time()
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
traing_time=end-start
conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

/home/jrech/predictive-goods-receipt-scheduling/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jrech/predictive-goods-receipt-scheduling/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Results for LightGBM saved successfully at 2026-06-03 11:29.


In [ ]:

Model="Multilayer Perceptron"
epochs=100
best_params=pd.read_sql(Model, engine).iloc[0].to_dict()
X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw, scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer") , feature_output="MLP")
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)

best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

batch_size=best_params.pop("batch_size")

train_dataset = data_utils.TensorDataset(X_tr, y_tr, w_tr)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataset = data_utils.TensorDataset(X_val, y_val, w_val)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


model=MLP_model.Model(input_dim=X_train.shape[1])

optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=best_params.pop("weight_decay"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, min_lr=1e-6)


start=time.time()
accuracay_plot=model.fit(train_dataloader, val_dataloader,epochs, optimizer, scheduler)
end=time.time()

y_pred=model.predict(X_test)
y_proba=model.predict_proba(X_test)

traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

Fold 1, Loss: 1.5482, Val Loss: 1.2957, Val Acc: 0.4796
Fold 2, Loss: 1.3733, Val Loss: 1.2436, Val Acc: 0.5081
Fold 3, Loss: 1.3038, Val Loss: 1.2007, Val Acc: 0.5062
Fold 4, Loss: 1.2502, Val Loss: 1.1741, Val Acc: 0.5220
Fold 5, Loss: 1.2297, Val Loss: 1.1529, Val Acc: 0.5330
Fold 6, Loss: 1.2008, Val Loss: 1.1211, Val Acc: 0.5512
Fold 7, Loss: 1.1688, Val Loss: 1.1118, Val Acc: 0.5411
Fold 8, Loss: 1.1569, Val Loss: 1.0884, Val Acc: 0.5578
Fold 9, Loss: 1.1283, Val Loss: 1.0688, Val Acc: 0.5694
Fold 10, Loss: 1.1175, Val Loss: 1.0539, Val Acc: 0.5806
Fold 11, Loss: 1.0954, Val Loss: 1.0379, Val Acc: 0.5831
Fold 12, Loss: 1.0871, Val Loss: 1.0183, Val Acc: 0.6012
Fold 13, Loss: 1.0670, Val Loss: 1.0123, Val Acc: 0.5881
Fold 14, Loss: 1.0526, Val Loss: 0.9999, Val Acc: 0.6040
Fold 15, Loss: 1.0427, Val Loss: 0.9897, Val Acc: 0.5996
Fold 16, Loss: 1.0334, Val Loss: 0.9769, Val Acc: 0.6192
Fold 17, Loss: 1.0104, Val Loss: 0.9686, Val Acc: 0.6120
Fold 18, Loss: 1.0062, Val Loss: 0.9531,

In [ ]:
Model="TabNet"
best_params=pd.read_sql(Model, engine).iloc[0].to_dict()

X_train, X_test, y_train, y_test, w_train, w_test, cat_idxs, cat_dims = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler="none", encoder="ordinal", periodic_transformer=False, feature_output="TabNet")
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)

scaler = best_params.pop("scaler")
encoder = best_params.pop("encoder")
periodic_transformer = best_params.pop("periodic_transformer")
batch_size = best_params.pop("batch_size")
optimizer_params=best_params.pop("lr")
step_size=best_params.pop("step_size")
scheduler_gamma=best_params.pop("scheduler_gamma")

best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")



model = TabNetClassifier(
    **best_params,
    optimizer_params=dict(lr=optimizer_params),
    scheduler_params=dict(step_size=step_size, gamma=scheduler_gamma),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    seed=42,
)

start=time.time()

model.fit(
    X_train=X_tr,
    y_train=y_tr,
    weights=w_tr,
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    eval_set=[(X_val, y_val)],
    eval_weights=[w_val],
    eval_metric=['accuracy'],
    max_epochs=300,
    patience=100,
    batch_size=batch_size,
    verbose=0
)
end=time.time()


X_train = np.nan_to_num(X_train, nan=0)
X_test = np.nan_to_num(X_test, nan=0)

y_pred=model.predict(X_test)
y_proba=model.predict_proba(X_test)

traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)


/home/jrech/predictive-goods-receipt-scheduling/.venv/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


TypeError: TabModel.fit() got an unexpected keyword argument 'cat_idxs'

In [ ]:

Model="XGBoost"

best_params=pd.read_sql(Model, engine).iloc[0].to_dict()

scaler = best_params.pop("scaler")
encoder = best_params.pop("encoder")
periodic_transformer = best_params.pop("periodic_transformer")
number_ouf_boost_rounds = best_params.pop("num_boost_round")

X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler=scaler,encoder=encoder,periodic_transformer=periodic_transformer)
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split( X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)



dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
dval   = xgb.DMatrix(X_val, label=y_val, weight=w_val)
dtest  = xgb.DMatrix(X_test, label=y_test, weight=w_test)

start=time.time()
bst = xgb.train(
    best_params,
    dtrain,
    num_boost_round=number_ouf_boost_rounds,
    evals=[(dval, "val")],
    early_stopping_rounds=30,
    verbose_eval=False
)
end=time.time()
traing_time=end-start
y_proba = bst.predict(dtest, output_margin=False)
y_pred = y_proba.argmax(axis=1)
end=time.time()


traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)

res.save_frames(Model, y_pred.astype(str), y_proba ,y_test, w_test, training_time=traing_time)





Results for XGBoost saved successfully at 2026-05-26 09:30.


In [ ]:
sqlite_path = "Database/DB_results.db"
engine = sa.create_engine("sqlite:///" + sqlite_path)
engine.connect()
df=conf.bootstrap_results.round(4)
df.to_sql("ConfidenceIntervals", con=engine, if_exists="replace", index=False)


35